In [1]:
!pip install datasets
!pip install tree_sitter==0.20.4
!pip install boto3
!pip install smart_open
!pip install torch==2.6.0
!pip install vllm

### SEED GATHERING GET CONTENT

In [2]:
import datasets
import os
import signal
from multiprocessing import Pool
from botocore import UNSIGNED
from botocore.config import Config
from utils import process_chunk

/mmfs1/project/phan/sj863/ds677_sj863/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
NUMWORKERS = os.cpu_count()

In [4]:
ds = datasets.load_dataset("bigcode/the-stack-v2", 
                           "C", cache_dir=f"../nk569/stack", 
                           streaming=True, split="train")

In [5]:
i = 25000
data_list = []
for example in ds:
    data_list.append(example)
    i -= 1
    if not i:
        break

ds = data_list

In [6]:
funs = set()
total_len = len(ds)
CHUNK_SIZE = 1000 * NUMWORKERS

print(f"Total length: {total_len}")
print(f"Chunk size: {CHUNK_SIZE}")

chunk = []
p = Pool(NUMWORKERS)

Total length: 25000
Chunk size: 128000


In [7]:
for i, ex in enumerate(iter(ds)):
    if i % (total_len // 100) == 0:
        print(f"{i}/{total_len}")
    try:
        chunk.append(ex)
        if len(chunk) == CHUNK_SIZE or i == total_len - 1:
            print(f"Processing chunk {i // CHUNK_SIZE}")
            # divide the chunk into NUM_WORKERS chunks
            subchunk_size = len(chunk) // NUMWORKERS
            subchunks = [chunk[i:i + subchunk_size]
                         for i in range(0, len(chunk), subchunk_size)]
            new_funs_iter = p.imap(
                process_chunk, [(i, subchunk) for i, subchunk in enumerate(subchunks)])
            print("Getting new functions")
            len_before = len(funs)
            while True:
                try:
                    def timeout_handler(_, __):
                        raise KeyboardInterrupt  # it's fineeeeeee
                    signal.signal(signal.SIGALRM, timeout_handler)
                    signal.alarm(240)
                    funs.update(next(new_funs_iter))
                    signal.alarm(0)
                except KeyboardInterrupt:
                    signal.alarm(0)
                    print("Keyboard interrupt. Terminating pool")
                    p.terminate()
                    p = Pool(NUMWORKERS)
                    break
                except StopIteration:
                    break
                except Exception as e:
                    print(e)

            signal.alarm(0)
            print(f"Done processing chunk {i // CHUNK_SIZE}. Got {len(funs) - len_before} new functions")

            chunk = []
    except Exception as e:
        print(e)
        chunk = []

    if i == total_len - 1:
        break


p.close()
new_ds_dict = {
    "content": list(funs),
    "id": list(range(len(funs)))
}

new_ds = datasets.Dataset.from_dict(new_ds_dict)


0/25000
250/25000
500/25000
750/25000
1000/25000
1250/25000
1500/25000
1750/25000
2000/25000
2250/25000
2500/25000
2750/25000
3000/25000
3250/25000
3500/25000
3750/25000
4000/25000
4250/25000
4500/25000
4750/25000
5000/25000
5250/25000
5500/25000
5750/25000
6000/25000
6250/25000
6500/25000
6750/25000
7000/25000
7250/25000
7500/25000
7750/25000
8000/25000
8250/25000
8500/25000
8750/25000
9000/25000
9250/25000
9500/25000
9750/25000
10000/25000
10250/25000
10500/25000
10750/25000
11000/25000
11250/25000
11500/25000
11750/25000
12000/25000
12250/25000
12500/25000
12750/25000
13000/25000
13250/25000
13500/25000
13750/25000
14000/25000
14250/25000
14500/25000
14750/25000
15000/25000
15250/25000
15500/25000
15750/25000
16000/25000
16250/25000
16500/25000
16750/25000
17000/25000
17250/25000
17500/25000
17750/25000
18000/25000
18250/25000
18500/25000
18750/25000
19000/25000
19250/25000
19500/25000
19750/25000
20000/25000
20250/25000
20500/25000
20750/25000
21000/25000
21250/25000
21500/25000
21

In [8]:
ds = new_ds
ds

Dataset({
    features: ['content', 'id'],
    num_rows: 6513
})

In [9]:
ds["content"][250]

'/**\n * xmlParseURIQuery:\n * @uri:  pointer to an URI structure\n * @str:  pointer to the string to analyze\n *\n * Parse the query part of an URI\n * \n * query = *uric\n *\n * Returns 0 or the error code\n */static int\nxmlParseURIQuery(xmlURIPtr uri, const char **str)\n{\n    const char *cur = *str;\n\n    if (str == NULL)\n        return (-1);\n\n    while (IS_URIC(cur) || ((uri != NULL) && (uri->cleanup) && (IS_UNWISE(cur))))\n        NEXT(cur);\n    if (uri != NULL) {\n        if (uri->query != NULL)\n            xmlFree(uri->query);\n        uri->query = xmlURIUnescapeString(*str, cur - *str, NULL);\n    }\n    *str = cur;\n    return (0);\n}'

In [10]:
save_dir = "../datasets/seed1"
new_ds.save_to_disk(save_dir)

Saving the dataset (1/1 shards): 100%|█| 6513/6513 [00:00<00:00, 280


### SEED GATHERING HIGH-QUALITY SUBSET

In [11]:
import subprocess
import tempfile
import signal
import hashlib
import os
import argparse
from typing import List, Dict
from tqdm import tqdm
from tree_sitter_parser import LANGUAGE, global_parser, does_have_return

In [12]:
# Function to run gcc and get error counts for each file
def run_gcc(d):
    try:
        outs = subprocess.run(
            ["gcc", "-Wall", "-o", "output", "*.c"],  # Use clang or gcc for compilation
            cwd=d,
            capture_output=True,
            timeout=120,
            text=True,
        ).stderr  # Capture stderr for errors and warnings
    except Exception as e:
        print(e)
        return None

    filemap = {}
    lines = outs.split("\n")
    for line in lines:
        if line.strip():
            parts = line.split(":")
            if len(parts) >= 3:  # GCC or clang usually outputs like file.c:line:error
                file = os.path.basename(parts[0])  # Get just the file name
                if file not in filemap:
                    filemap[file] = 0  # Initialize error count
                if "error" in line:  # Look for error messages
                    filemap[file] += 1

    return filemap

# Function to typecheck a batch of C files
def typecheck_batch(files: List[str]) -> Dict[str, str]:
    # Create a temporary directory using the tempfile module
    filemap: Dict[str, str] = {}
    with tempfile.TemporaryDirectory() as tempdir:
        for contents in files:
            # Hash the content to generate unique file names
            hash_object = hashlib.sha1(bytes(contents, "utf8"))
            hex_dig = hash_object.hexdigest()
            filemap[hex_dig] = contents  # Mapping hashed names to content
            name = os.path.join(tempdir, hex_dig + ".c")  # Save file as a .c file
            with open(name, "w") as f:
                f.write(contents)

        # Run gcc in the temporary directory
        typecheck_map = run_gcc(tempdir)
        print(typecheck_map)

        if typecheck_map is None:
            return {}

        for contents, errors in typecheck_map.items():
            no_c = contents.replace(".c", "")  # Remove .c extension to match hashed names
            if errors == 0:
                continue
            if no_c in filemap:
                del filemap[no_c]  # Remove files with errors from the map

        print(f"Pass rate: {len(filemap)}/{len(files)}")
        return filemap


In [13]:
print("Filtering to only functions with return statements")
ds = ds.filter(lambda ex: does_have_return(ex["content"]), num_proc=os.cpu_count())


Filtering to only functions with return statements


Filter (num_proc=128): 100%|█| 6513/6513 [00:03<00:00, 2102.17 examp


In [14]:
batch = []
max_i = len(ds) - 1

new_ds = {
    "content": [],
    "sha1": [],
    "id": [],
}

e_id = 0

for i, ex in enumerate(tqdm(ds, total=len(ds))):
    try:
        code = ex["content"]

        batch.append(code)

        if len(batch) == 250 or i == max_i:
            filemap = typecheck_batch(batch)
            for sha1, contents in filemap.items():
                new_ds["content"].append(contents)
                new_ds["sha1"].append(sha1)
                new_ds["id"].append(e_id)
                e_id += 1
            batch = []

    except Exception as e:
        print(f"There was an error: {e}")
        continue

new_ds_hf = datasets.Dataset.from_dict(new_ds)

 16%|████                     | 1000/6242 [00:00<00:01, 3312.71it/s]

{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250


 42%|██████████▌              | 2645/6242 [00:00<00:00, 6040.70it/s]

{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250


 80%|████████████████████     | 5018/6242 [00:00<00:00, 7014.71it/s]

{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250


100%|█████████████████████████| 6242/6242 [00:01<00:00, 6087.69it/s]

{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 250/250
{'cc1': 1}
Pass rate: 242/242


In [15]:
print(new_ds_hf['content'][3400])

/**
  * @brief  Calculate K.
  * @param  
  * @retval 
  */static uint16_t Get_64K(uint16_t Ax, uint16_t Ay, uint16_t Bx, uint16_t By)
{
	uint16_t Tmp1, Tmp2;
	Tmp1 = By - Ay;
	Tmp2 = Bx - Ax;
	return (Tmp1 * 50 / Tmp2);
}


In [16]:
save_dir = "../datasets/seed2"
new_ds_hf.save_to_disk(save_dir)

Saving the dataset (1/1 shards): 100%|█| 6242/6242 [00:00<00:00, 717


In [17]:
new_ds_hf

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 6242
})

### SEED GATHERING FILTER DATASET

In [18]:
import datasets
import os
from tree_sitter_parser import global_parser, LANGUAGE, does_have_return, make_parser
import benchmark_data
from tqdm import tqdm
import torch
import argparse
from vllm import LLM, SamplingParams
import random
import re

INFO 05-16 20:47:34 [__init__.py:239] Automatically detected platform cuda.


2025-05-16 20:47:40,041	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [19]:
def template_few_shot(code, answer, rationale):
    doc, code = c_extract_docstring(code)
    assert answer == "No" or answer == "Yes"
    prompt = f"""<issue_start>username_0: I have a function written in C and I'd like someone to check my description of this function.
I'm doing this so that I can write a good description for this function.


Here is the code for the function:
```c
{code}
```

Here is my description of this program::
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with "Yes" or "No" depending on if my description has enough information alone to re-implement the function.
Also, answer with "No" if the description does not match the function.<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is: {answer}

{rationale}

Upvotes: 200"""
    return prompt


FEW_SHOTS = [
    ('''/**
     * Squares all elements in a matrix.
     *
     */
    void square_matrix(int matrix[][MAX_COLS]) {
        int rows = sizeof(matrix) / sizeof(matrix[0]);
        int cols = sizeof(matrix[0]) / sizeof(matrix[0][0]);
    
        for (int i = 0; i < rows; i++) {
            for (int j = 0; j < cols; j++) {
                matrix[i][j] = matrix[i][j] * matrix[i][j];
            }
        }
    }
    ''',
     "Yes",
     "Following the standard description for docstrings for functions and methods, the square_matrix function description tells exactly what the function does."
    ),
    (
        '''/**
     * @brief Reverses a singly linked list.
     *
     * @param head The head of the linked list to be reversed.
     * @return The new head of the reversed linked list.
     * Note:
     * - If the list is empty (i.e., `head` is NULL), the function will return NULL.
     * - If the list has only one node, the list will remain unchanged.
     */
    struct Node* reverseLinkedList(struct Node* head) {
        struct Node *prev = NULL, *current = head, *next = NULL;

        while (current != NULL) {
            next = current->next;
            current->next = prev;
            prev = current;
            current = next;
        }
        head = prev;
        return head;
    }''',
        "Yes",
        "The docstring for the reverseLinkedList function is very detailed and describes its purpose and also specifies the parameters needed to reverse a singly linked list.",
    ),

    (
        '''/**
     * Function to convert Celsius to Fahrenheit
     *
     */
        float celsius_to_fahrenheit(float celsius) {
            return (celsius * 9.0 / 5.0) + 32.0;
        }''',
        "Yes",
        "The doscstring does seem to match the implementation! The function simply converts celsius to fahrenheit as explained.",
    ),

    (
        '''/**
     * Computes the dot product of two vectors.
     *
     * This function calculates the dot product of two vectors by multiplying
     * corresponding elements from both vectors and sums the results to return the dot product.
     */
    double dot_product(const double* vec1, const double* vec2, size_t size) {
        double result = 0.0;
        for (size_t i = 0; i < size; i++) {
            result += vec1[i] * vec2[i];
        }
        return result;
    }''',
    "Yes",
    "dot_product is a very simple function, the docstring explains its purpose, which is computing the dot product of two vectors.",
    ),

    ('''/**
     * Pushes an element onto the stack.
     * 0 on success, -1 if the stack is full.
     */
    int push(Stack *stack, int value) {
        if (stack->top == MAX_STACK_SIZE - 1) {
            printf("Stack overflow\n");
            return -1;
        }
        stack->data[++stack->top] = value;
        return 0;
    }
    ''',
     "Yes",
     "The docstring for the push function describes the function's purpose and gives detailed information.",
     )
]


def prompt_fmt(code):
    doc, code = c_extract_docstring(code)
    random.shuffle(FEW_SHOTS)
    buf = ""
    for few in FEW_SHOTS:
        buf += template_few_shot(*few)
    buf += f"""<issue_start>username_0: I have a function written in C and I'd like someone to check my description of this function.
I'm doing this so that I can write a good description for this function.

Here is the code for the function:
```c
{code}
```

Here is my description of this program:
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with "Yes" or "No" depending on if my description has enough information to re-implement the function.
Also, answer with "No" if the description does not match the function.
Upvotes: 100<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is:"""
    return buf


def auto_dtype():
    if torch.cuda.is_bf16_supported():
        return "bfloat16"
    return "auto"


def chunkify(lst, n):
    chunks = []
    for i in range(0, len(lst), n):
        chunk = []
        for j in range(n):
            if i + j < len(lst):
                chunk.append(lst[i + j])
        chunks.append(chunk)
    return chunks


In [20]:
dataset = new_ds_hf

In [21]:
print(f"Loaded {len(dataset)} examples. Running pre-filtering...")

BAD_WORDS = ["todo", "fixme", "bug"]
BAD_IMPORTS = ["unistd.h", "signal.h", "sys/wait.h"]
BAD_IMPORTS = [f"#include <{b}>" for b in BAD_IMPORTS]
BAD_SUBSTRINGS = BAD_WORDS + BAD_IMPORTS

bench_filter = benchmark_data.filter_out()
all_bench = bench_filter["human_eval_docstrings"] + \
    bench_filter["human_eval_solutions"] + \
    bench_filter["mbpp_docstrings"] + \
    bench_filter["mbpp_solutions"]

Loaded 6242 examples. Running pre-filtering...
num strings from mbpp_docstrings: 120
num strings from mbpp_solutions: 120
num strings from human_eval_docstrings: 164
num strings from human_eval_solutions: 161


In [22]:
TOPLEVEL_DOCSTRING_QUERY = LANGUAGE.query("""
(
    (comment) @docstring .
    (function_definition) @function.def
    (#match? @docstring "^/\\\*\\\*\\\n" )
)
""")

def pre_filtering(ex):
    code = ex["content"]
    code_bytes = code.encode('utf-8')

    # filter out bad substrings
    lower = code.lower()
    for word in BAD_SUBSTRINGS:
        if word in lower:
            return False

    for b in all_bench:
        if b in code:  # contaminated sample!
            return False

    # too many lines of code -- say 150
    lines = code.split("\n")
    if len(lines) > 150:
        return False
    parser = make_parser()
    if not does_have_return(code, parser=parser):
        return False

    try:
        tree = global_parser.parse(code_bytes)
        docstring, _ = TOPLEVEL_DOCSTRING_QUERY.captures(tree.root_node)[0]

        # get the docstring, filter if not a docstring
        docstring_text = docstring.text.decode('utf-8')
        if not docstring_text.startswith('/**\n') and not docstring_text.endswith('*/'):
            return False
    except Exception as e:
        print(f"Error in filtering: {e}")
        return False

    return True  # all good!


threads = os.cpu_count() - 1  # type: ignore
dataset = dataset.filter(pre_filtering, num_proc=threads)

/mmfs1/project/phan/sj863/ds677_sj863/lib/python3.10/site-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '_ctypes.PyCFuncPtrType'>.
  StockPickler.save(self, obj, save_persistent_id)
/mmfs1/project/phan/sj863/ds677_sj863/lib/python3.10/site-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '_ctypes.PyCFuncPtrType'>: _ctypes.PyCFuncPtrType has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
Parameter 'function'=<function pre_filtering at 0x15538c4de200> of the transform datasets.arrow_dataset.Dataset.filter@2.0.1 couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hash

In [23]:
dataset

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 5493
})

In [24]:
model = LLM(f"/project/phan/codellama/StarCoder", dtype=auto_dtype(),
            gpu_memory_utilization=0.95, tensor_parallel_size=1)
tokenizer = model.get_tokenizer()

INFO 05-16 20:47:51 [config.py:2968] Downcasting torch.float32 to torch.bfloat16.
INFO 05-16 20:48:10 [config.py:717] This model supports multiple tasks: {'embed', 'generate', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
INFO 05-16 20:48:10 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 05-16 20:48:12 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 05-16 20:48:26 [__init__.py:239] Automatically detected platform cuda.
INFO 05-16 20:48:32 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='/project/phan/codellama/StarCoder', speculative_config=None, tokenizer='/project/phan/codellama/StarCoder', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, toke

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:00<00:08,  1.52it/s]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:01<00:08,  1.48it/s]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:02<00:07,  1.48it/s]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:02<00:06,  1.55it/s]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:03<00:05,  1.53it/s]
Loading safetensors checkpoint shards:  43% Completed | 6/14 [00:03<00:05,  1.49it/s]
Loading safetensors checkpoint shards:  50% Completed | 7/14 [00:04<00:04,  1.46it/s]
Loading safetensors checkpoint shards:  57% Completed | 8/14 [00:05<00:04,  1.46it/s]
Loading safetensors checkpoint shards:  64% Completed | 9/14 [00:06<00:03,  1.47it/s]
Loading safetensors checkpoint shards:  71% Completed | 10/14 [00:08<00:04,  1.08s/it]
Loading safetensors checkpoint shards:  79% Completed | 11/14

INFO 05-16 20:48:46 [loader.py:458] Loading weights took 12.16 seconds
INFO 05-16 20:48:47 [gpu_model_runner.py:1347] Model loading took 29.7279 GiB and 12.474039 seconds
INFO 05-16 20:49:06 [backends.py:420] Using cache directory: /home/sj863/.cache/vllm/torch_compile_cache/0ca5a55d42/rank_0_0 for vLLM's torch.compile
INFO 05-16 20:49:06 [backends.py:430] Dynamo bytecode transform time: 19.00 s
INFO 05-16 20:49:14 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 8.205 s
INFO 05-16 20:49:28 [monitor.py:33] torch.compile takes 19.00 s in total
INFO 05-16 20:49:29 [kv_cache_utils.py:634] GPU KV cache size: 564,944 tokens
INFO 05-16 20:49:29 [kv_cache_utils.py:637] Maximum concurrency for 16,384 tokens per request: 34.48x
INFO 05-16 20:49:49 [gpu_model_runner.py:1686] Graph capturing finished in 21 secs, took 0.59 GiB
INFO 05-16 20:49:49 [core.py:159] init engine (profile, create kv cache, warmup model) took 62.95 seconds
INFO 05-16 20:49:50 [core_

In [25]:
print(f"Now running stage 3 filtering on {len(dataset)} examples...")

Now running stage 3 filtering on 5493 examples...


In [26]:
def unindent(s):
    lines = s.splitlines()
    non_blank_lines = [line for line in lines if line.strip()]
    min_indent = min((len(line) - len(line.lstrip())) for line in non_blank_lines) if non_blank_lines else 0

    processed_lines = []
    for line in lines:
        stripped = line.lstrip()
        if stripped.startswith('@'):
            stripped = re.sub(r'^\s*@\w+\s*', '', stripped)

        processed_lines.append(stripped)

    return '\n'.join(processed_lines)



def c_extract_docstring(code):
    first_doc = code.find('/**')
    assert first_doc != -1
    first_doc = first_doc + 3
    second_doc = code[first_doc+1:].find('*/')
    assert second_doc != -1
    second_doc = second_doc + first_doc + 1
    doc = code[first_doc:second_doc]
    doc = unindent(doc).strip()
    code = code[second_doc+2:]
    return doc, code

In [27]:
dummy = '/**\n*/ int dummy() { return 0;}'
dummy_prompt = prompt_fmt(dummy)
few_shot_toks = len(tokenizer.encode(
    dummy_prompt)) - len(tokenizer.encode(dummy))
print(f"Few-shot prompt has {few_shot_toks} tokens")

Few-shot prompt has 1645 tokens


In [28]:
dataset["content"][200]

'/**\n   @brief Executes the "Shutdown" command to shut down the ZigBee interface.\n\n   @param Parameter_Count is number of elements in Parameter_List.\n   @param Parameter_List is list of parsed arguments associate with this\n          command.\n\n   @return\n    - QCLI_STATUS_SUCCESS_E indicates the command is executed successfully.\n    - QCLI_STATUS_ERROR_E   indicates the command is failed to execute.\n    - QCLI_STATUS_USAGE_E   indicates there is usage error associated with this\n                            command.\n*/QCLI_Command_Status_t qc_api_cmd_ZB_Shutdown(uint32_t Parameter_Count, QCLI_Parameter_t *Parameter_List)\n{\n   QCLI_Command_Status_t Ret_Val;\n\n   /* Verify the ZigBee layer had been initialized. */\n   if(ZigBee_Demo_Context.ZigBee_Handle != NULL)\n   {\n      /* Cleanup the clusters have have been created. */\n      //ZB_Cluster_Cleanup();\n\n      /* Shutdown the ZigBee stack. */\n      qc_drv_ZB_Shutdown(qc_api_get_qc_drv_context(), ZigBee_Demo_Context.ZigB

In [29]:
dataset["content"][0]

'/**\n   @brief Executes the "AddGroup" command to add a group.\n\n   Parameter_List[0] ID of the device to send the command to.\n   Parameter_List[1] Endpoint of the Groups client cluster to use to send the\n                     command.\n   Parameter_List[2] ID of the group to add.\n   Parameter_List[3] Name of the group to add.\n   Parameter_List[4] Flag indicating if groups should be added only if the\n                     device is identifying.\n\n   @param Parameter_Count is number of elements in Parameter_List.\n   @param Parameter_List is list of parsed arguments associate with this\n          command.\n\n   @return\n    - QCLI_STATUS_SUCCESS_E indicates the command is executed successfully.\n    - QCLI_STATUS_ERROR_E indicates the command is failed to execute.\n    - QCLI_STATUS_USAGE_E indicates there is usage error associated with this\n      command.\n*/QCLI_Command_Status_t qc_api_cmd_ZCL_Groups_AddGroup(uint32_t Parameter_Count, QCLI_Parameter_t *Parameter_List)\n{\n   QC

In [ ]:
prompts = []
for ex in tqdm(dataset, total=len(dataset), desc="Generating prompts"):
    code = ex["content"]
    toks = len(tokenizer.encode(code)) + few_shot_toks
    if toks > 16380:
        print(f"Skipping example with {toks} tokens")
        # to skip, just add dummy prompt
        prompts.append(dummy_prompt)
        continue
    p = prompt_fmt(code)
    prompts.append(p)

responses = []
for chunk in tqdm(chunkify(prompts, 512), desc="Generating responses"):
    outs = model.generate(chunk, SamplingParams(
        temperature=0.0, stop="\n", max_tokens=5))
    contents = [o.outputs[0].text for o in outs]
    for c in contents:
        yes_count = c.lower().count("yes")
        no_count = c.lower().count("no")
        if yes_count > no_count:
            responses.append(True)
        elif yes_count < no_count:
            responses.append(False)
        else:
            # default to No
            responses.append(False)

Generating responses:   0%|                  | 0/11 [00:00<?, ?it/s]
Processed prompts:   0%| | 0/512 [00:00<?, ?it/s, est. speed input: 
Processed prompts:   0%| | 1/512 [00:00<06:15,  1.36it/s, est. speed
Processed prompts:   0%| | 2/512 [00:02<12:53,  1.52s/it, est. speed
Processed prompts:   2%| | 11/512 [00:04<03:10,  2.62it/s, est. spee
Processed prompts:   5%| | 26/512 [00:06<01:46,  4.58it/s, est. spee
Processed prompts:   8%| | 39/512 [00:09<01:31,  5.19it/s, est. spee
Processed prompts:  10%| | 51/512 [00:11<01:25,  5.36it/s, est. spee
Processed prompts:  14%|▏| 72/512 [00:13<01:03,  6.88it/s, est. spee
Processed prompts:  18%|▏| 93/512 [00:15<00:53,  7.84it/s, est. spee
Processed prompts:  22%|▏| 114/512 [00:17<00:47,  8.46it/s, est. spe
Processed prompts:  27%|▎| 138/512 [00:19<00:40,  9.31it/s, est. spe
Processed prompts:  33%|▎| 167/512 [00:21<00:32, 10.58it/s, est. spe
Processed prompts:  39%|▍| 198/512 [00:24<00:26, 11.73it/s, est. spe
Processed prompts:  44%|▍| 227/512

In [ ]:
dataset

In [ ]:
new_ds = dataset.filter(  # horrible hack!
    lambda ex, i: responses[i] and '/**\n*/ int dummy() { return 0;}' not in ex["content"], with_indices=True)
print(f"Filtered {len(dataset) - len(new_ds)} examples")

In [ ]:
new_ds

In [ ]:
new_ds.save_to_disk("../datasets/seed3")